# Physical posteriors for one Gaia DR4 microlensing event

This worked example starts with one Gaia `source_id`, reads its row from the
Gaia DR4 microlensing table, samples the published fitted parameters, and
derives posterior distributions for:

- microlensing parallax and lens mass;
- source and lens distance;
- source, lens-source-relative, and lens proper motion; and
- the lens transverse velocity.

The calculations use deterministic Monte Carlo propagation of the catalogue's
marginal fitted errors. Because the full fit covariance matrix is unavailable,
the inputs are sampled independently from Gaussians. This propagates uncertainty
in the Gaia microlensing solution; no Galactic mass or distance prior is applied.

## How to use this notebook

1. Run notebook 02 to create `outputs/gaiadr4/gaia_dr4_microlensing_events_joined.csv`.
2. Change only `SOURCE_ID` in the configuration cell below.
3. Run all cells from top to bottom.

The identifier is kept as a string while reading CSV files.  Gaia source IDs
are 64-bit integers and can be silently rounded if software converts them to
floating point.


The data are described in the Gaia DR4 publication by **Wyrzykowski et al. (2026)**.

In [ ]:
# Standard scientific-Python dependencies only; no archive login is needed
# once notebook 02 has produced the local joined table.
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display



def find_project_root(start):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "01_archive_access.ipynb").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from the gaia-dr4-microlensing-notebooks folder.")

ROOT = find_project_root(Path.cwd())

# Put Matplotlib's cache in this writable project, which also works in a
# restricted or temporary Jupyter environment.
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".matplotlib_cache"))
import matplotlib.pyplot as plt

# -------------------------------------------------------------------------
# USER CONFIGURATION: normally only SOURCE_ID needs to change.
# -------------------------------------------------------------------------
# Example source; use an exact decimal string.
SOURCE_ID = "6059400613544951552"
TABLE_PATH = ROOT / "outputs" / "gaiadr4" / "gaia_dr4_microlensing_events_joined.csv"
EVENT_NAMES_PATH = ROOT / "data" / "event_names.csv"
SOURCE_ID_COLUMN = "source_id"

# The final study scripts used 100,000 draws for mass/distance and 50,000
# for kinematics, both with a source-ID-derived seed.  Keeping these values
# unchanged makes repeated runs deterministic and numerically reproduces
# the saved production posterior summaries.
N_MASS_DISTANCE_DRAWS = 100_000
N_KINEMATICS_DRAWS = 50_000
RANDOM_SEED = int(SOURCE_ID) % (2**32 - 1)

# Physical constants/conversion factors used by the study.
KAPPA_MAS_PER_MSUN = 8.144       # kappa = 4G/(c^2 AU), in mas / solar mass
DAYS_PER_YEAR = 365.25
KMS_PER_MASYR_KPC = 4.74047      # 1 mas/yr at 1 kpc, in km/s

# Figures are shown inline.  Set this to True to save PNG copies beside the
# notebook outputs; existing files for the same source ID will be replaced.
SAVE_FIGURES = False
FIGURE_DIR = ROOT / "outputs" / "gaiadr4" / "single_event_posteriors"

## Equations and conventions

The fitted parallax vector is

$$\pi_E = \sqrt{\pi_{E,E}^2 + \pi_{E,N}^2},$$

and the relative lens-source parallax, lens mass, and lens parallax are

$$\pi_{\rm rel}=\theta_E\pi_E,\qquad
  M_L=\frac{\theta_E}{\kappa\pi_E},\qquad
  \varpi_L=\varpi_S+\pi_{\rm rel}.$$

Since parallax is in milliarcseconds, its reciprocal is directly a distance
in kiloparsecs: $D\,[\mathrm{kpc}]=1/\varpi\,[\mathrm{mas}]$.  We invert
every accepted parallax draw and then take percentiles; we never invert only
the central value.

The relative proper-motion magnitude and components are

$$\mu_{\rm rel}=365.25\frac{\theta_E}{t_E},\qquad
  (\mu_{{\rm rel},\alpha *},\mu_{{\rm rel},\delta})=
  \mu_{\rm rel}\frac{(\pi_{E,E},\pi_{E,N})}{\pi_E}.$$

Here `pmra` means $\mu_{\alpha *}=\dot\alpha\cos\delta$ (eastward), while
`pmdec` is northward.  Following the study,

$$\boldsymbol\mu_L=\boldsymbol\mu_S+\boldsymbol\mu_{\rm rel},\qquad
  \boldsymbol v_{L,\perp}=4.74047\,D_L\boldsymbol\mu_L.$$

All quoted intervals below are the 16th, 50th, and 84th percentiles.  The
physical masks retain positive $\theta_E$ and, where required, positive
$t_E$, source parallax, or lens parallax.  Rejected-draw fractions are shown
explicitly rather than hidden.


In [ ]:
# Only these 14 columns are needed.  Reading a small subset makes this cell
# fast even if TABLE_PATH is later changed to a much wider public table.
PARAMETER_COLUMNS = {
    "theta_E": ("paczynski3_einstein_radius", "paczynski3_einstein_radius_error", "mas"),
    "t_E": ("paczynski2_te", "paczynski2_te_error", "day"),
    "pi_E_east": ("paczynski2_parallax_east", "paczynski2_parallax_east_error", ""),
    "pi_E_north": ("paczynski2_parallax_north", "paczynski2_parallax_north_error", ""),
    "source_parallax": ("paczynski3_varpi_source", "paczynski3_varpi_source_error", "mas"),
    "source_pmra": ("paczynski3_pmra_source", "paczynski3_pmra_source_error", "mas/yr"),
    "source_pmdec": ("paczynski3_pmdec_source", "paczynski3_pmdec_source_error", "mas/yr"),
}

columns_to_read = [SOURCE_ID_COLUMN]
for value_column, error_column, _ in PARAMETER_COLUMNS.values():
    columns_to_read.extend([value_column, error_column])

catalogue = pd.read_csv(
    TABLE_PATH,
    usecols=columns_to_read,
    dtype={SOURCE_ID_COLUMN: "string"},
)
catalogue[SOURCE_ID_COLUMN] = catalogue[SOURCE_ID_COLUMN].str.strip()
selected = catalogue.loc[catalogue[SOURCE_ID_COLUMN].eq(str(SOURCE_ID))]

if len(selected) != 1:
    raise LookupError(
        f"Expected exactly one row for source_id={SOURCE_ID}; found {len(selected)} "
        f"in {TABLE_PATH}."
    )
row = selected.iloc[0]

# The event-name table is convenient but is not needed for the physics.
# Therefore a future user can run this with only the Gaia joined table.
event_name = "(event-name table unavailable)"
if EVENT_NAMES_PATH.is_file():
    event_names = pd.read_csv(EVENT_NAMES_PATH, dtype={"sourceid": "string", "source_id": "string"})
    name_id = "source_id" if "source_id" in event_names else "sourceid"
    name_match = event_names.loc[event_names[name_id].str.strip().eq(str(SOURCE_ID))]
    if len(name_match) == 1:
        event_name = str(name_match.iloc[0]["event"])

print(f"Event:     {event_name}")
print(f"source_id: {SOURCE_ID}")
print(f"Input:     {TABLE_PATH}")

In [ ]:
# Extract every value/error pair, convert it explicitly to float, and fail
# early with an informative message if this event cannot support the full
# calculation.  Zero uncertainty is allowed; negative uncertainty is not.
fit = {}
input_rows = []
for short_name, (value_column, error_column, unit) in PARAMETER_COLUMNS.items():
    value = float(row[value_column])
    error = float(row[error_column])
    if not np.isfinite(value):
        raise ValueError(f"{value_column} is missing/non-finite for source_id={SOURCE_ID}.")
    if not np.isfinite(error) or error < 0:
        raise ValueError(f"{error_column} must be a finite, non-negative uncertainty.")
    fit[short_name] = (value, error)
    input_rows.append((short_name, value_column, value, error, unit))

input_table = pd.DataFrame(
    input_rows,
    columns=["parameter", "Gaia DR4 column", "fit value", "reported 1-sigma error", "unit"],
)
display(input_table)


## 1. Mass and distance posterior

This block intentionally follows the study's mass/distance script, including
its random-number seed, 100,000 draws, and draw order:
source parallax, Einstein radius, east parallax component, north parallax
component.  Positive-$\theta_E$ draws define the mass posterior.  Lens
distance additionally requires positive lens parallax.


In [ ]:
# A fresh generator is important: the production mass/distance script also
# starts from this seed, independently of the later kinematics calculation.
rng_mass_distance = np.random.default_rng(RANDOM_SEED)

md_source_parallax = rng_mass_distance.normal(*fit["source_parallax"], N_MASS_DISTANCE_DRAWS)
md_theta_E = rng_mass_distance.normal(*fit["theta_E"], N_MASS_DISTANCE_DRAWS)
md_pi_E_east = rng_mass_distance.normal(*fit["pi_E_east"], N_MASS_DISTANCE_DRAWS)
md_pi_E_north = rng_mass_distance.normal(*fit["pi_E_north"], N_MASS_DISTANCE_DRAWS)

md_pi_E = np.hypot(md_pi_E_east, md_pi_E_north)
md_lens_parallax = md_source_parallax + md_theta_E * md_pi_E

# These masks deliberately match the production script: positive theta_E
# defines the physical-parameter/mass set, and positive lens parallax adds
# the distance requirement.  For continuous Gaussian component draws,
# pi_E = 0 has probability zero.
md_mass_mask = md_theta_E > 0
md_distance_mask = md_mass_mask & (md_lens_parallax > 0)

lens_mass = md_theta_E[md_mass_mask] / (KAPPA_MAS_PER_MSUN * md_pi_E[md_mass_mask])
lens_distance = 1.0 / md_lens_parallax[md_distance_mask]

# Keep paired mass-distance draws for the joint diagnostic plot later.
paired_lens_mass = md_theta_E[md_distance_mask] / (
    KAPPA_MAS_PER_MSUN * md_pi_E[md_distance_mask]
)

print(f"Positive-theta_E mass draws:        {md_mass_mask.mean():8.3%}")
print(f"Positive lens-parallax draws:       {md_distance_mask.sum() / md_mass_mask.sum():8.3%}")
print(f"Accepted lens-distance draws:       {md_distance_mask.sum():8,d} / {N_MASS_DISTANCE_DRAWS:,d}")
if md_mass_mask.sum() < 100 or md_distance_mask.sum() < 100:
    raise RuntimeError("Fewer than 100 physical mass/distance draws; posterior is not reportable.")


## 2. Source/lens motion and transverse-velocity posterior

This second, independently seeded block follows the study's kinematics
script and its draw order: Einstein radius, timescale, source parallax,
parallax-vector components, and source proper-motion components.  Its lens
distance draws are kept correlated with lens motion when computing velocity.
The separately sampled `lens_distance` above remains the study's reported
lens-distance posterior.


In [ ]:
rng_kinematics = np.random.default_rng(RANDOM_SEED)

k_theta_E = rng_kinematics.normal(*fit["theta_E"], N_KINEMATICS_DRAWS)
k_t_E = rng_kinematics.normal(*fit["t_E"], N_KINEMATICS_DRAWS)
k_source_parallax = rng_kinematics.normal(*fit["source_parallax"], N_KINEMATICS_DRAWS)
k_pi_E_east = rng_kinematics.normal(*fit["pi_E_east"], N_KINEMATICS_DRAWS)
k_pi_E_north = rng_kinematics.normal(*fit["pi_E_north"], N_KINEMATICS_DRAWS)
source_pmra = rng_kinematics.normal(*fit["source_pmra"], N_KINEMATICS_DRAWS)
source_pmdec = rng_kinematics.normal(*fit["source_pmdec"], N_KINEMATICS_DRAWS)

k_pi_E = np.hypot(k_pi_E_east, k_pi_E_north)
k_lens_parallax = k_source_parallax + k_theta_E * k_pi_E

source_distance_mask = (k_source_parallax > 0) & np.isfinite(k_source_parallax)
kinematic_mask = (
    (k_theta_E > 0)
    & (k_t_E > 0)
    & (k_pi_E > 0)
    & (k_lens_parallax > 0)
    & np.isfinite(k_lens_parallax)
)

source_distance = 1.0 / k_source_parallax[source_distance_mask]
source_pm = np.hypot(source_pmra, source_pmdec)

# All arrays below share kinematic_mask, preserving correlations through
# the nonlinear transformations into proper motion and velocity.
relative_pm = DAYS_PER_YEAR * k_theta_E[kinematic_mask] / k_t_E[kinematic_mask]
relative_pmra = relative_pm * k_pi_E_east[kinematic_mask] / k_pi_E[kinematic_mask]
relative_pmdec = relative_pm * k_pi_E_north[kinematic_mask] / k_pi_E[kinematic_mask]

lens_pmra = source_pmra[kinematic_mask] + relative_pmra
lens_pmdec = source_pmdec[kinematic_mask] + relative_pmdec
lens_pm = np.hypot(lens_pmra, lens_pmdec)

# This kinematic distance must be used here (rather than independently
# pairing the reported D_L posterior) to retain its covariance with mu_L.
k_lens_distance = 1.0 / k_lens_parallax[kinematic_mask]
lens_v_east = KMS_PER_MASYR_KPC * lens_pmra * k_lens_distance
lens_v_north = KMS_PER_MASYR_KPC * lens_pmdec * k_lens_distance
lens_v_transverse = np.hypot(lens_v_east, lens_v_north)

print(f"Positive source-parallax draws:     {source_distance_mask.mean():8.3%}")
print(f"Joint physical kinematics draws:    {kinematic_mask.mean():8.3%}")
print(f"Accepted joint kinematics draws:    {kinematic_mask.sum():8,d} / {N_KINEMATICS_DRAWS:,d}")
if source_distance_mask.sum() < 100 or kinematic_mask.sum() < 100:
    raise RuntimeError("Fewer than 100 physical distance/kinematic draws; posterior is not reportable.")


## 3. Numerical posterior summaries

`error_minus` and `error_plus` are the distances from the median to the
16th and 84th percentiles.  They are intentionally asymmetric because
inverse-parallax distances and vector magnitudes are generally non-Gaussian.


In [ ]:
def posterior_summary(quantity, draws, unit, description):
    '''Summarize one finite posterior with the study's percentile convention.'''
    draws = np.asarray(draws)
    draws = draws[np.isfinite(draws)]
    if len(draws) < 100:
        raise RuntimeError(f"Only {len(draws)} finite draws for {quantity}.")
    p16, median, p84 = np.percentile(draws, [16, 50, 84])
    return {
        "quantity": quantity,
        "description": description,
        "unit": unit,
        "p16": p16,
        "median": median,
        "p84": p84,
        "error_minus": median - p16,
        "error_plus": p84 - median,
        "n_draws": len(draws),
    }


posterior_items = [
    (r"$\pi_E$", md_pi_E, "", "microlensing-parallax magnitude"),
    (r"$M_L$", lens_mass, r"$M_\odot$", "lens mass"),
    (r"$D_S$", source_distance, "kpc", "source distance"),
    (r"$D_L$", lens_distance, "kpc", "lens distance"),
    (r"$\mu_S$", source_pm, "mas/yr", "source proper-motion magnitude"),
    (r"$\mu_{S,\alpha *}$", source_pmra, "mas/yr", "source eastward proper motion"),
    (r"$\mu_{S,\delta}$", source_pmdec, "mas/yr", "source northward proper motion"),
    (r"$\mu_{rel}$", relative_pm, "mas/yr", "lens-source relative proper-motion magnitude"),
    (r"$\mu_{rel,\alpha *}$", relative_pmra, "mas/yr", "relative eastward proper motion"),
    (r"$\mu_{rel,\delta}$", relative_pmdec, "mas/yr", "relative northward proper motion"),
    (r"$\mu_L$", lens_pm, "mas/yr", "lens proper-motion magnitude"),
    (r"$\mu_{L,\alpha *}$", lens_pmra, "mas/yr", "lens eastward proper motion"),
    (r"$\mu_{L,\delta}$", lens_pmdec, "mas/yr", "lens northward proper motion"),
    (r"$v_{L,E}$", lens_v_east, "km/s", "lens eastward transverse velocity"),
    (r"$v_{L,N}$", lens_v_north, "km/s", "lens northward transverse velocity"),
    (r"$v_{L,\perp}$", lens_v_transverse, "km/s", "lens transverse-speed magnitude"),
]

summary = pd.DataFrame(
    posterior_summary(quantity, draws, unit, description)
    for quantity, draws, unit, description in posterior_items
)
display(summary.style.format({
    "p16": "{:.5g}", "median": "{:.5g}", "p84": "{:.5g}",
    "error_minus": "{:.5g}", "error_plus": "{:.5g}", "n_draws": "{:,d}",
}))

# `summary` is left as a normal DataFrame for downstream reuse/export.
summary


## 4. One-dimensional posterior plots

The shaded band is the central 68% interval and the black line is the
median.  Histograms are clipped to the 0.5th–99.5th percentile plotting
window so a handful of near-zero parallax draws cannot compress the useful
part of a distance/velocity panel.  **The summaries above use every accepted
draw; clipping is visual only.**


In [ ]:
def draw_posterior(ax, draws, label, unit, color="#2878b5"):
    '''Draw a robustly ranged density histogram with its 16/50/84 markers.'''
    finite = np.asarray(draws)[np.isfinite(draws)]
    plot_low, plot_high = np.percentile(finite, [0.5, 99.5])
    shown = finite[(finite >= plot_low) & (finite <= plot_high)]
    p16, median, p84 = np.percentile(finite, [16, 50, 84])

    ax.hist(shown, bins=70, range=(plot_low, plot_high), density=True,
            color=color, alpha=0.78, edgecolor="none")
    ax.axvspan(p16, p84, color=color, alpha=0.18, label="16th–84th percentile")
    ax.axvline(median, color="black", lw=1.4, label="median")
    ax.set_xlabel(f"{label} [{unit}]" if unit else label)
    ax.set_ylabel("posterior density")
    ax.grid(alpha=0.18, lw=0.6)


scalar_panels = [
    (md_pi_E, r"$\pi_E$", ""),
    (lens_mass, r"$M_L$", r"$M_\odot$"),
    (source_distance, r"$D_S$", "kpc"),
    (lens_distance, r"$D_L$", "kpc"),
    (source_pm, r"$\mu_S$", "mas/yr"),
    (relative_pm, r"$\mu_{rel}$", "mas/yr"),
    (lens_pm, r"$\mu_L$", "mas/yr"),
    (lens_v_transverse, r"$v_{L,\perp}$", "km/s"),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 7), constrained_layout=True)
for ax, (draws, label, unit) in zip(axes.flat, scalar_panels):
    draw_posterior(ax, draws, label, unit)
fig.suptitle(f"{event_name}  |  Gaia source_id {SOURCE_ID}", fontsize=14)

if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURE_DIR / f"{SOURCE_ID}_scalar_posteriors.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print(f"Saved {path}")
plt.show()


## 5. Joint/vector posterior plots

These panels reveal correlations that one-dimensional intervals cannot.
The first three show the sampled two-component vectors.  The last uses
paired draws from the mass/distance calculation and is especially useful
for spotting a long inverse-parallax distance tail.


In [ ]:
def robust_limits(x, y):
    '''Return 0.5–99.5% limits independently in x and y for readable plots.'''
    return (*np.percentile(x[np.isfinite(x)], [0.5, 99.5]),
            *np.percentile(y[np.isfinite(y)], [0.5, 99.5]))


vector_panels = [
    (source_pmra, source_pmdec, r"$\mu_{S,\alpha *}$ [mas/yr]",
     r"$\mu_{S,\delta}$ [mas/yr]", "Source proper motion"),
    (lens_pmra, lens_pmdec, r"$\mu_{L,\alpha *}$ [mas/yr]",
     r"$\mu_{L,\delta}$ [mas/yr]", "Lens proper motion"),
    (lens_v_east, lens_v_north, r"$v_{L,E}$ [km/s]",
     r"$v_{L,N}$ [km/s]", "Lens transverse velocity"),
    (paired_lens_mass, lens_distance, r"$M_L$ [$M_\odot$]",
     r"$D_L$ [kpc]", "Paired mass and lens distance"),
]

fig, axes = plt.subplots(2, 2, figsize=(11, 9), constrained_layout=True)
for ax, (x, y, xlabel, ylabel, title) in zip(axes.flat, vector_panels):
    x = np.asarray(x); y = np.asarray(y)
    xlo, xhi, ylo, yhi = robust_limits(x, y)
    visible = (x >= xlo) & (x <= xhi) & (y >= ylo) & (y <= yhi)
    density = ax.hexbin(x[visible], y[visible], gridsize=55, mincnt=1, cmap="viridis")
    ax.axvline(np.median(x), color="white", lw=0.9, alpha=0.8)
    ax.axhline(np.median(y), color="white", lw=0.9, alpha=0.8)
    ax.set(xlim=(xlo, xhi), ylim=(ylo, yhi), xlabel=xlabel, ylabel=ylabel, title=title)
    fig.colorbar(density, ax=ax, label="draws per hexagon")

fig.suptitle(f"Joint posteriors: {event_name}", fontsize=14)
if SAVE_FIGURES:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURE_DIR / f"{SOURCE_ID}_joint_posteriors.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    print(f"Saved {path}")
plt.show()


## Interpretation and limitations

- The source parallax and source proper motion used here are the
  **Paczyński-3 microlensing-fit parameters**, not the ordinary
  `gaia_source.parallax`, `pmra`, and `pmdec` columns.
- Marginal fit errors are sampled as independent Gaussians because the
  covariance matrix is not published in this table.  Correlations created
  by the derived equations are retained within each Monte-Carlo block.
- Truncating to positive physical draws changes a posterior when the fitted
  value is close to zero.  Inspect the printed accepted fractions before
  interpreting such an event.  The study required at least 100 accepted
  draws to report a posterior; this notebook applies the same minimum.
- No distance prior, stellar-population prior, extinction model, or lens
  mass-function prior is applied.  Very broad inverse-parallax posteriors
  should not be mistaken for precise distance measurements.
- The direction of relative motion is taken directly from the fitted
  $\boldsymbol\pi_E$ vector, and the lens/source vectors are combined in the
  fitted convention used by the study.  No additional Earth-velocity frame
  correction is applied here.
- `lens_v_transverse` is a two-dimensional transverse speed; it is not a
  three-dimensional space velocity and contains no radial-velocity term.

For machine-readable reuse, `input_table` contains the seven fitted inputs
and `summary` contains all percentile results.  For example:

```python
summary.to_csv(f"{SOURCE_ID}_physical_posterior_summary.csv", index=False)
```
